# fact_6 · Fonte를 웹사이트에 연결하는 임시 서버 (캐글)
캐글 노트북에서 Fonte(Gemma 4 E4B + LoRA `fonte`)를 켜고, 임시 https 주소를 만들어요. 사이트 관리자 페이지에 **주소와 비밀번호**를 넣으면 LAGO 사이트의 Fonte 모드가 이 모델로 답해요.

**설정:** GPU T4 x1 이상 · Internet **On** · Add-ons → Secrets에 `HF_TOKEN` 체크 (`MODEL_KEY`를 Secrets에 넣어 두면 매번 같은 비밀번호를 써요)

**주의**
- 이 노트북이 켜져 있는 동안만 작동해요 (캐글은 최대 9~12시간, 끄면 끝). 다시 켜면 주소가 바뀌어요 → 관리자에 다시 입력.
- `MODEL_NAME`은 `fact_2_학습.ipynb`의 값과 **똑같이** 맞춰 주세요.
- 이 노트북은 캐글에서 아직 실행해 보지 못한 상태예요. 오류가 나면 로그를 알려 주세요.

In [ ]:
!pip -q install fastapi uvicorn nest_asyncio
!pip -q install unsloth
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

In [ ]:
import os, secrets
from kaggle_secrets import UserSecretsClient
sec = UserSecretsClient()
HF_TOKEN = sec.get_secret("HF_TOKEN")
try:
    KEY = sec.get_secret("MODEL_KEY")          # Secrets에 넣어 두면 매번 같은 비밀번호
except Exception:
    KEY = secrets.token_urlsafe(24)            # 없으면 이번에만 쓰는 새 비밀번호

MODEL_NAME = "unsloth/gemma-4-E4B-it"   # ← fact_2_학습.ipynb 의 MODEL_NAME 과 똑같이!
LORA_REPO  = "lagem/LAGO_AI_MODELS"
LORA_SUB   = "fonte"
MAX_SEQ    = 2048

In [ ]:
from unsloth import FastModel
from peft import PeftModel
model, tokenizer = FastModel.from_pretrained(model_name=MODEL_NAME, max_seq_length=MAX_SEQ, load_in_4bit=True, token=HF_TOKEN)
model = PeftModel.from_pretrained(model, LORA_REPO, subfolder=LORA_SUB, token=HF_TOKEN)
FastModel.for_inference(model)
print("모델 준비 완료")

In [ ]:
import threading, torch
from fastapi import FastAPI, Header, HTTPException
from pydantic import BaseModel
app = FastAPI()
lock = threading.Lock()
tok = getattr(tokenizer, "tokenizer", tokenizer)

class Req(BaseModel):
    system: str
    user: str
    max_new_tokens: int = 400

@app.get("/health")
def health():
    return {"ok": True}

@app.post("/generate")
def generate(r: Req, x_key: str = Header(default="")):
    if not secrets.compare_digest(x_key, KEY):
        raise HTTPException(401, "bad key")
    msgs = [{"role": "system", "content": r.system}, {"role": "user", "content": r.user}]
    ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True).to("cuda")
    with lock, torch.no_grad():
        out = model.generate(**ids, max_new_tokens=min(r.max_new_tokens, 600), do_sample=True, temperature=0.2, top_p=0.9)
    text = tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    return {"answer": text}

In [ ]:
import uvicorn, nest_asyncio, subprocess, re, time, requests
nest_asyncio.apply()
threading.Thread(target=lambda: uvicorn.run(app, host="127.0.0.1", port=8000, log_level="warning"), daemon=True).start()
time.sleep(3)
p = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"], stderr=subprocess.PIPE, text=True)
URL = None
for line in p.stderr:
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
    if m:
        URL = m.group(0); break
print("=" * 60)
print("관리자 페이지 → 키와 설정에 아래 두 값을 넣으세요")
print("모델 서버 주소 (MODEL_URL):", URL)
print("모델 서버 비밀번호 (MODEL_KEY):", KEY)
print("=" * 60)
print("건강 확인:", requests.get(URL + "/health", timeout=30).text)

In [ ]:
# 이 셀을 계속 켜 두세요 (멈추면 사이트에서 답이 안 와요)
import time
while True:
    time.sleep(300)
    print("살아 있어요", time.strftime("%H:%M:%S"))